![FLIP Banner](../../Assets/images/flip-banner.png)

# FLIP: Agentic AI in Practice
**Module 08: Advanced Agentic AI**

---

## Session 8D: MCP Fundamentals — Tool and Context Servers

<div align="center">

<table>
<thead><tr><th><strong>Item</strong></th><th><strong>Description</strong></th></tr></thead>
<tbody>
<tr><td align="left">Estimated time</td><td>2 hours</td></tr>
<tr><td align="left">Mandatory part</td><td>Local mock MCP-style client/server workflow</td></tr>
<tr><td align="left">Optional part</td><td>Real MCP SDK exploration if available</td></tr>
<tr><td align="left">Main output</td><td>A mock MCP server exposing tools, resources and prompts with capability checks</td></tr>
</tbody>
</table>

</div>

### Table of Contents

1. Overview and Learning Goals  
2. Conceptual Background  
3. Setup  
4. MCP-Style Server Manifest  
5. Local MCP Client/Server Workflow  
6. Inspection and Capability Boundaries  
7. Optional Real MCP SDK Section  
8. Testing and Analysis  
9. Student Tasks  
10. Submission and Reflection

### 1. Overview and Learning Goals

MCP, or Model Context Protocol, is an advanced integration concept for agentic AI. Earlier notebooks directly defined tools inside a Python notebook. That is useful for learning, but real agent systems often need a more portable way to connect AI applications to tools and context.

A simplified MCP-style architecture is:

```mermaid
flowchart LR
    A[AI application / host] --> B[MCP client]
    B --> C[MCP server]
    C --> D[Tools]
    C --> E[Resources]
    C --> F[Prompts]
```

This notebook does not require a real MCP server. It uses a local mock server so you can understand the structure: capability discovery, resources, prompts, tool calls, argument validation and rejection of unsupported capabilities.

### 2. Conceptual Background

MCP-style systems separate **capability discovery** from **capability invocation**. Discovery tells a client what the server claims to provide. Invocation is the actual tool call or resource read. These are different stages and both require controls.

Core concepts:

<div align="center">

<table>
<thead><tr><th><strong>Concept</strong></th><th><strong>Meaning</strong></th><th><strong>Risk</strong></th></tr></thead>
<tbody>
<tr><td align="left">Tool</td><td>A callable action such as search, calculate or create ticket.</td><td>May create side effects if poorly scoped.</td></tr>
<tr><td align="left">Resource</td><td>Readable context such as a document, schema or note.</td><td>May leak private data if exposed incorrectly.</td></tr>
<tr><td align="left">Prompt</td><td>A reusable instruction template.</td><td>May encode unsafe or over-broad behaviour.</td></tr>
<tr><td align="left">Manifest</td><td>A description of exposed capabilities.</td><td>Not a security guarantee by itself.</td></tr>
</tbody>
</table>

</div>

A standard interface does not automatically make an agent safe. The server still needs argument validation, capability scoping, logging and refusal for unknown or unsafe calls.

In [ ]:
import json
import re
from typing import Any, Dict, List

print("M08D MCP setup complete.")

### 4. MCP-Style Server Manifest

The manifest below is a local teaching version of an MCP server description. It lists tools, resources and prompts. The client can discover these capabilities, but the server must still validate every request.

In [ ]:
MCP_SERVER_MANIFEST = {
    "server_name": "flip_teaching_mcp_server",
    "version": "0.1.0",
    "tools": {
        "rectangle_area": {
            "description": "Calculate rectangle area from non-negative width and height.",
            "required_args": ["width", "height"],
            "risk": "low"
        },
        "search_public_notes": {
            "description": "Search approved public teaching notes.",
            "required_args": ["query"],
            "risk": "low"
        },
    },
    "resources": {
        "resource://unit/syllabus": {
            "description": "Synthetic public unit syllabus summary.",
            "risk": "low"
        },
        "resource://unit/safety-rules": {
            "description": "Synthetic public safety rules.",
            "risk": "low"
        },
    },
    "prompts": {
        "safe_context_answer": {
            "description": "Answer using only approved context.",
            "variables": ["question", "context"]
        }
    }
}

APPROVED_RESOURCES = {
    "resource://unit/syllabus": "FLIP covers foundations, Flowise, LangChain, RAG, LangGraph, multi-agent systems, safety, model adaptation and advanced agents.",
    "resource://unit/safety-rules": "Do not use private data, hidden instructor materials, credentials, shell commands, email sending or unsafe external side effects."
}

PUBLIC_NOTES = [
    "MCP-style servers expose tools, resources and prompts to clients.",
    "A tool call should be validated before execution.",
    "Resources provide readable context but should not contain private data.",
    "Prompts can be reused as workflow templates.",
]

print(json.dumps(MCP_SERVER_MANIFEST, indent=2))

### 5. Local MCP Client/Server Workflow

The mock server implements four behaviours:

```text
1. list_capabilities()
2. read_resource(uri)
3. get_prompt(name)
4. call_tool(tool_name, args)
```

The server, not the client, enforces the final validation. This is important because a client may be buggy, compromised or too permissive.

In [ ]:
def rectangle_area(width: float, height: float) -> float:
    return width * height

def word_count(text: str) -> int:
    return len(text.split())

def validate_rectangle_args(args: Dict[str, Any]) -> Dict[str, Any]:
    if not isinstance(args, dict):
        return {"ok": False, "error": "arguments must be a dictionary", "result": None}
    for key in ["width", "height"]:
        if key not in args:
            return {"ok": False, "error": f"missing argument: {key}", "result": None}
        try:
            value = float(args[key])
        except (TypeError, ValueError):
            return {"ok": False, "error": f"{key} must be numeric", "result": None}
        if value < 0:
            return {"ok": False, "error": f"{key} must be non-negative", "result": None}
        args[key] = value
    return {"ok": True, "error": None, "result": args}

def validate_word_count_args(args: Dict[str, Any]) -> Dict[str, Any]:
    if not isinstance(args, dict) or "text" not in args:
        return {"ok": False, "error": "missing text argument", "result": None}
    if not isinstance(args["text"], str) or not args["text"].strip():
        return {"ok": False, "error": "text must be a non-empty string", "result": None}
    return {"ok": True, "error": None, "result": {"text": args["text"]}}

def search_public_notes(query: str) -> List[str]:
    query_terms = set(re.findall(r"[a-zA-Z_]+", query.lower()))
    results = []
    for note in PUBLIC_NOTES:
        note_terms = set(re.findall(r"[a-zA-Z_]+", note.lower()))
        if query_terms.intersection(note_terms):
            results.append(note)
    return results

In [ ]:
class MockMCPServer:
    def __init__(self, manifest):
        self.manifest = manifest

    def list_capabilities(self):
        return {"ok": True, "error": None, "result": self.manifest}

    def read_resource(self, uri: str):
        if uri not in self.manifest["resources"]:
            return {"ok": False, "error": f"unknown resource: {uri}", "result": None}
        if uri not in APPROVED_RESOURCES:
            return {"ok": False, "error": f"resource unavailable: {uri}", "result": None}
        return {"ok": True, "error": None, "result": {"uri": uri, "content": APPROVED_RESOURCES[uri]}}

    def get_prompt(self, name: str):
        if name not in self.manifest["prompts"]:
            return {"ok": False, "error": f"unknown prompt: {name}", "result": None}
        template = "Question: {question}\nContext: {context}\nAnswer using only approved context."
        return {"ok": True, "error": None, "result": {"name": name, "template": template}}

    def call_tool(self, tool_name: str, args: Dict[str, Any]):
        if tool_name not in self.manifest["tools"]:
            return {"ok": False, "error": f"unknown tool: {tool_name}", "result": None}

        if tool_name == "rectangle_area":
            validation = validate_rectangle_args(dict(args))
            if not validation["ok"]:
                return validation
            return {"ok": True, "error": None, "result": {"tool": tool_name, "value": rectangle_area(**validation["result"])}}

        if tool_name == "search_public_notes":
            query = args.get("query") if isinstance(args, dict) else None
            if not isinstance(query, str) or not query.strip():
                return {"ok": False, "error": "query must be a non-empty string", "result": None}
            return {"ok": True, "error": None, "result": {"tool": tool_name, "value": search_public_notes(query)}}

        if tool_name == "word_count":
            validation = validate_word_count_args(args)
            if not validation["ok"]:
                return validation
            return {"ok": True, "error": None, "result": {"tool": tool_name, "value": word_count(validation["result"]["text"])}}

        return {"ok": False, "error": f"tool not implemented: {tool_name}", "result": None}

In [ ]:
class MockMCPClient:
    def __init__(self, server):
        self.server = server

    def discover(self):
        return self.server.list_capabilities()

    def call_tool(self, tool_name: str, args: Dict[str, Any]):
        return self.server.call_tool(tool_name, args)

    def answer_with_resource(self, question: str, resource_uri: str):
        resource = self.server.read_resource(resource_uri)
        if not resource["ok"]:
            return resource
        prompt = self.server.get_prompt("safe_context_answer")
        if not prompt["ok"]:
            return prompt
        return {
            "ok": True,
            "error": None,
            "result": {
                "question": question,
                "resource_uri": resource_uri,
                "answer": "Based on the approved resource, " + resource["result"]["content"],
                "limitations": ["This answer uses only the selected approved resource."]
            }
        }

server = MockMCPServer(MCP_SERVER_MANIFEST)
client = MockMCPClient(server)

print(client.call_tool("rectangle_area", {"width": 3, "height": 4}))
print(client.answer_with_resource("What does the unit cover?", "resource://unit/syllabus"))

### 6. Inspection and Capability Boundaries

Inspect MCP-style responses for:

```text
1. Was the capability known?
2. Was the resource approved?
3. Were arguments valid?
4. Was the result limited to approved data?
5. Did unknown tools fail safely?
```

In [ ]:
def display_response(response):
    if not response.get("ok"):
        print("ERROR:", response.get("error"))
    else:
        print(json.dumps(response["result"], indent=2))

for response in [
    client.call_tool("rectangle_area", {"width": 5, "height": 2}),
    client.call_tool("delete_all_files", {}),
    client.call_tool("rectangle_area", {"width": -5, "height": 2}),
    client.call_tool("word_count", {"text": "MCP exposes tools resources prompts"}),
    client.answer_with_resource("What are the safety rules?", "resource://unit/safety-rules"),
]:
    print("\n---")
    display_response(response)

### 7. Optional Real MCP SDK Section

This section is optional. A real MCP SDK can replace the mock client/server, but the safety principles are unchanged:

```text
1. Expose only necessary capabilities.
2. Validate every argument.
3. Avoid private resources unless authorised.
4. Log calls.
5. Reject unknown or high-risk capabilities.
```

Do not connect to an untrusted MCP server in this lab.

In [ ]:
def optional_real_mcp_available():
    return False

if not optional_real_mcp_available():
    print("Skipped: real MCP SDK not configured in this environment.")

### 8. Testing and Analysis

In [ ]:
manifest_response = client.discover()
assert manifest_response["ok"] is True
assert "tools" in manifest_response["result"]

area = client.call_tool("rectangle_area", {"width": 6, "height": 7})
assert area["ok"] is True
assert area["result"]["value"] == 42

negative = client.call_tool("rectangle_area", {"width": -1, "height": 7})
assert negative["ok"] is False

unknown = client.call_tool("send_email", {"to": "someone@example.com"})
assert unknown["ok"] is False

resource = server.read_resource("resource://unit/safety-rules")
assert resource["ok"] is True
assert "private data" in resource["result"]["content"]

unknown_resource = server.read_resource("resource://private/grades")
assert unknown_resource["ok"] is False

wc = client.call_tool("word_count", {"text": "one two three"})
assert wc["ok"] is True
assert wc["result"]["value"] == 3

print("All M08D mandatory MCP-style tests passed.")

### 9. Student Tasks

<div align="center">

<table>
<thead><tr><th><strong>Task</strong></th><th><strong>What to do</strong></th><th><strong>Evidence</strong></th></tr></thead>
<tbody>
<tr><td align="left">Run baseline tests</td><td>Run all mandatory cells.</td><td>Test output.</td></tr>
<tr><td align="left">Add one safe tool</td><td>Add or modify a deterministic low-risk tool.</td><td>Manifest and server code.</td></tr>
<tr><td align="left">Validate arguments</td><td>Reject missing and invalid arguments.</td><td>Validation examples.</td></tr>
<tr><td align="left">Add tests</td><td>Test valid call, invalid arguments and unknown capability.</td><td>Assert-based tests.</td></tr>
<tr><td align="left">Analyse security boundary</td><td>Explain why dangerous tools should not be exposed.</td><td>Short paragraph.</td></tr>
</tbody>
</table>

</div>

### 10. Submission and Reflection

Submit:

```text
1. Baseline test output.
2. Added safe tool and updated manifest.
3. Validation logic.
4. Added tests.
5. Security-boundary paragraph.
6. Optional real MCP output or skipped note.
7. 150–250 word reflection.
```

Further readings:

- MCP official introduction: https://modelcontextprotocol.io/docs/getting-started/intro
- MCP specification: https://modelcontextprotocol.io/specification/2025-06-18
- MCP Python SDK: https://github.com/modelcontextprotocol/python-sdk
- OpenAI Agents SDK MCP guide: https://openai.github.io/openai-agents-python/mcp/
- LangChain MCP adapters: https://docs.langchain.com/oss/python/langchain/mcp